# HEP Paper Suite — Google Colab (Free T4)
Runs the full upgraded-HEP experiment suite on Colab's CUDA T4. Free tier: 12h limit, ~12GB VRAM — enough for all ResNet9 batches (113 MB). For CIFAR-100/MobileNet, keep batch 32.

**Repo:** `https://github.com/nam200718/Topology-aware-FDL.git` (pushed `f25560d` contains all fixes)

Run cells top-to-bottom. Outputs persist to Drive if mounted, otherwise download the `outputs/` zip at the end.

## 0. (Optional) Mount Drive for persistence across 12h restarts

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# Pick a Drive folder for outputs, e.g. /content/drive/MyDrive/hep_outputs
# Later, symlink: !ln -s /content/drive/MyDrive/hep_outputs /content/Topology-aware-FDL/outputs

## 1. Clone repo (or pull if already cloned)

In [ ]:
import os, pathlib
REPO = "https://github.com/nam200718/Topology-aware-FDL.git"
ROOT = "/content/Topology-aware-FDL"
if not pathlib.Path(ROOT).exists():
    !git clone $REPO $ROOT
else:
    !git -C $ROOT pull --rebase
%cd $ROOT
!git log --oneline -3

## 2. Install dependencies (CUDA PyTorch is preinstalled on Colab — keep it)

In [ ]:
# Colab ships torch with CUDA; don't downgrade to cpu build.
# Install only the missing deps from requirements.txt (skip torch if present).
!pip -q install -r requirements.txt
!python -c "import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')"

## 3. (Optional) Symlink outputs to Drive so a 12h timeout doesn't lose data

In [ ]:
import os, pathlib
DRIVE_OUT = "/content/drive/MyDrive/hep_outputs"  # change if you mounted elsewhere
if pathlib.Path("/content/drive").exists():
    os.makedirs(DRIVE_OUT, exist_ok=True)
    # Move any existing outputs, then symlink
    if os.path.exists("outputs") and not os.path.islink("outputs"):
        !mv outputs outputs_local 2>/dev/null; echo "moved old outputs"
    if not os.path.islink("outputs"):
        !ln -s $DRIVE_OUT outputs
        print(f"outputs -> {DRIVE_OUT}")
else:
    print("Drive not mounted — outputs stay ephemeral. Download the zip at the end.")

## 4. Quick fix check (5 min) — proves the two crash fixes work on CUDA

In [ ]:
!python scripts/run_comparison.py --config configs/quick_fix_check.yaml
print("quick fix check done — if no RuntimeError, mixed-size and head-explosion fixes are good on CUDA")

## 5. Run the full paper suite (resume-safe, ~15h total — run in chunks if needed)

The driver skips completed blocks and resumes failed ones. If Colab times out at 12h, just **Runtime → Restart and rerun** from this cell — it picks up where it left off.

**What it runs:** `freeze_cells` → `ablation` (+ moderate complement) → `ksweep` → `convergence` → `table3_baselines` (45 runs, 3 seeds) → `comparison` seeds 123/7 (40 runs) → `byz` grids → `cifar100` (10) → `scale50` (6) → `mobilenet` (4). Progress: `outputs/paper_suite/manifest.json`

In [ ]:
!python scripts/run_paper_suite.py
print("suite finished (or paused at 12h limit — rerun this cell to resume)")

### 5b. If you only need the main Table III (headline 3-seed), run just that:

In [ ]:
# Core-4 methods × 5 regimes × 3 seeds — the paper's headline table
!python scripts/run_paper_suite.py --only table3_baselines comparison_seed123 comparison_seed7
# Or even quicker single-seed preview:
# !python scripts/run_comparison.py --config configs/comparison.yaml

## 6. Generate all paper figures from collected artifacts

In [ ]:
!python scripts/make_paper_figures.py
!ls -lh paper/figures/*.png | tail -n +1

## 7. Assemble Table III Mean±Std + significance tests (once 3-seed data exists)

In [ ]:
!python scripts/build_table3.py --out paper/table3_generated.tex
!cat paper/table3_generated.tex

# Paired t-tests (HEP vs baselines, per scenario):
# Pass every multi_seed raw CSV plus the seed-42 comparison CSV with --seed tags as needed.
# Example (adjust paths to your actual run dirs):
# !python scripts/compute_significance.py \
#     --raw outputs/multi_seed_eval_*/all_seeds_raw.csv \
#     --raw outputs/comparison_study_*/comparison_results.csv --seed 42 \
#     --out outputs/paper_suite/significance_tests.csv

## 8. Download results (if not using Drive symlink)

Colab's disk is ephemeral — zip outputs before the runtime recycles.

In [ ]:
!zip -r /tmp/hep_outputs.zip outputs/paper_suite paper/table3_generated.tex paper/figures/*.png 2>/dev/null | tail -1
from google.colab import files
files.download("/tmp/hep_outputs.zip")